<a href="https://colab.research.google.com/github/jrennr/Neural_Network_Charity_Analysis/blob/main/Copy_of_Copy_of_AlphabetSoupCharity_Jenn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

data_path = Path('charity_data.csv')
source = data_path if data_path.is_file() else 'https://static.bc-edx.com/data/dl-1-2/m21/lms/starter/charity_data.csv'
application_df = pd.read_csv(source)
application_df = application_df.drop(columns=['EIN', 'NAME'], errors='ignore').dropna(subset=['IS_SUCCESSFUL'])
X = application_df.drop(columns='IS_SUCCESSFUL')
y = application_df['IS_SUCCESSFUL'].astype('int64')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train = X_train.copy()
X_test = X_test.copy()
for column in ['APPLICATION_TYPE', 'CLASSIFICATION']:
    counts = X_train[column].value_counts()
    retained = counts[counts >= 100].index
    X_train[column] = X_train[column].where(X_train[column].isin(retained), 'Other')
    X_test[column] = X_test[column].where(X_test[column].isin(retained), 'Other')


In [ ]:
categorical_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_columns = X_train.columns.difference(categorical_columns).tolist()
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', encoder)]), categorical_columns)
])
X_train_scaled = preprocessor.fit_transform(X_train).astype('float32')
X_test_scaled = preprocessor.transform(X_test).astype('float32')


In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
nn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(15, activation='relu'),
    tf.keras.layers.Dense(7, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
nn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
nn_model.summary()


In [ ]:
fit_model = nn_model.fit(X_train_scaled, y_train, epochs=5, batch_size=32)
model_loss, model_accuracy = nn_model.evaluate(X_test_scaled, y_test, verbose=2)
print(f'Loss: {model_loss}, Accuracy: {model_accuracy}')
